# 20 — HOG + SVM (Nguyên) — KHUNG

**Việc của Nguyên trước khi chạy:**
1. Cài đặt `HogExtractor` trong `src/banana_ripeness/features/hog.py` (các chỗ `TODO(Nguyên)`).
2. Chọn bộ phân loại + lưới tham số trong `src/banana_ripeness/models/hog_svm.py`.
3. Khai báo thí nghiệm trong `configs/experiments.yaml` (mục `nguyen`).
4. Push code lên GitHub, rồi chạy notebook này trên Kaggle (cell setup sẽ `git pull` bản mới).

Phần chia dữ liệu, chọn tham số bằng CV theo nhóm, độ đo, bootstrap std, lưu kết quả và bảng so sánh
**đã có sẵn** và dùng chung với SVM màu của Luân (xem `models/svm_color.py` làm ví dụ).

In [ ]:
# === SETUP: chạy cell này đầu tiên (Kaggle / Colab / máy) ===
# Kaggle: bật Settings -> Internet = On. Code lấy từ GitHub, dữ liệu lấy từ Kaggle Dataset (Add Input).
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/LuanLuKim/cs231-banana-ripeness.git"
BRANCH = "main"
OWNER = "nguyen"          # tên người chạy, ghi vào bảng kết quả chung

local = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "banana_ripeness").exists()), None)
if local is not None:                       # chạy trên máy, trong thư mục repo
    CODE_DIR = local
else:                                       # Kaggle / Colab: clone vào /tmp (không lẫn vào output notebook)
    CODE_DIR = Path("/tmp/cs231-banana-ripeness")
    if (CODE_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(CODE_DIR), "pull", "-q", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE_DIR)], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
os.environ["BANANA_OWNER"] = OWNER
commit = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code:", CODE_DIR, "| commit", commit or "?")

%load_ext autoreload
%autoreload 2

In [ ]:
from banana_ripeness.data import find_data_root, load_manifest, get_split
import pandas as pd
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)

# Tự tìm thư mục có manifest.csv trong /kaggle/input (Kaggle Dataset output preprocessing đã Add Input).
# Nếu có nhiều bản, chỉ rõ: DATA_ROOT = find_data_root("/kaggle/input/<ten-dataset>")
DATA_ROOT = find_data_root()
df = load_manifest(DATA_ROOT)
EXP = CODE_DIR / "configs" / "experiments.yaml"
print("Dữ liệu:", DATA_ROOT, "|", len(df), "ảnh")
pd.crosstab([df.split_mixed], df.label).loc[["train", "val", "test"]]

In [ ]:
# Kiểm tra nhanh đặc trưng trên 1 ảnh trước khi chạy cả lưới
from banana_ripeness.models import build_model
from banana_ripeness.data import read_rgb, read_mask
m = build_model("hog_svm")
r = df.iloc[0]
f = m.extractor(read_rgb(r.image_path), read_mask(r.mask_path))
print("Số chiều HOG:", f.shape, "| key cache:", m.extractor.key)

In [ ]:
from banana_ripeness.experiment import load_experiments, run_grid, load_results, compare_table, plot_comparison
res = run_grid(load_experiments(EXP, "nguyen"), df=df, data_root=DATA_ROOT, owner=OWNER)
res = load_results(); res = res[res.owner == OWNER]
display(compare_table(res, split="test", scheme="mixed"))
display(compare_table(res, split="external", scheme="external"))

In [ ]:
from banana_ripeness.experiment import best_run, prediction_grid
best = best_run(res, scheme="mixed", by_split="val")
best.plot_confusion("test");
prediction_grid(best.predictions["test"], n=8, true="Half_Ripe", pred="Ripe", data_root=DATA_ROOT);